In [1]:
# 1) Mount Drive (if not yet)
# -------------------------
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [11]:

import os, shutil, json, datetime, joblib, numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from itertools import cycle

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16, ResNet50, MobileNetV2
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg_pre
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_pre
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_pre
from tensorflow.keras import mixed_precision

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize, StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline

In [3]:
# ---------------- PATH SETUP (your paths kept) ----------------
import datetime
DRIVE_ROOT = "/content/drive/MyDrive/Colab Notebooks"
SPLIT70_DIR = f"{DRIVE_ROOT}/emotion/emotion_preprocessed_split70_30_aug4000"
TEST_DIR = f"{DRIVE_ROOT}/emotion/emotion_preprocessed/test"

LOCAL_TRAIN = "/content/data/train"
LOCAL_VAL   = "/content/data/val"
LOCAL_TEST  = "/content/data/test"

OUT_DIR = f"{DRIVE_ROOT}/RF_Finetune_Optimized_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}"
os.makedirs(OUT_DIR, exist_ok=True)

# ---------------- COPY DATA ----------------
def fresh_copy(src, dst):
    if os.path.exists(dst):
        shutil.rmtree(dst)
    shutil.copytree(src, dst)

fresh_copy(os.path.join(SPLIT70_DIR, "train"), LOCAL_TRAIN)
fresh_copy(os.path.join(SPLIT70_DIR, "val"), LOCAL_VAL)
fresh_copy(TEST_DIR, LOCAL_TEST)
print("✅ Copied train/val/test to Colab local.")


✅ Copied train/val/test to Colab local.


In [19]:
  # ---------------- GLOBAL CONFIG (Speed + Regularization) ----------------
SEED = 42

# T4-friendly speedups
mixed_precision.set_global_policy('mixed_float16')  # feature extraction faster
IMG_SIZE = (160, 160)        # faster than 224, good enough features
BATCH_SIZE = 64              # tune if OOM
BACKBONE = "mobilenetv2"     # 'mobilenetv2' | 'resnet50' | 'vgg16'

# RF regularization (your requested changes)
MAX_DEPTH = 30               # between 20–40 (tweakable)
MIN_SAMPLES_LEAF = 3         # 3–5
MAX_FEATURES = "log2"        # per your ask
MIN_SAMPLES_SPLIT = 4

# RF sweep + early stop (for the single-RF curve)
N_GRID = [100, 200, 300, 400, 500, 600]
PATIENCE = 2

# Predict speed
PRED_WORKERS = 4
PRED_USE_MP = True

In [13]:
# ---------------- BACKBONE SELECT ----------------
if BACKBONE.lower() == "vgg16":
    BASE = VGG16; PREP = vgg_pre
elif BACKBONE.lower() == "resnet50":
    BASE = ResNet50; PREP = resnet_pre
elif BACKBONE.lower() == "mobilenetv2":
    BASE = MobileNetV2; PREP = mobilenet_pre
else:
    raise ValueError("BACKBONE must be 'vgg16', 'resnet50', or 'mobilenetv2'")


In [14]:
# ---------------- DATA PIPE ----------------
train_datagen = ImageDataGenerator(preprocessing_function=PREP)
val_datagen   = ImageDataGenerator(preprocessing_function=PREP)
test_datagen  = ImageDataGenerator(preprocessing_function=PREP)

train_gen = train_datagen.flow_from_directory(
    LOCAL_TRAIN, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False, seed=SEED
)
val_gen = val_datagen.flow_from_directory(
    LOCAL_VAL, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False, seed=SEED
)
test_gen = test_datagen.flow_from_directory(
    LOCAL_TEST, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False, seed=SEED
)

class_indices = train_gen.class_indices
idx_to_class = {v:k for k,v in class_indices.items()}
classes_sorted = [idx_to_class[i] for i in range(len(idx_to_class))]
num_classes = len(classes_sorted)


Found 28000 images belonging to 7 classes.
Found 8616 images belonging to 7 classes.
Found 7178 images belonging to 7 classes.


In [15]:
# ---------------- FEATURE EXTRACTOR (GAP) ----------------
base_model = BASE(include_top=False, weights="imagenet", input_shape=IMG_SIZE + (3,))
gap = tf.keras.layers.GlobalAveragePooling2D(dtype="float32")(base_model.output)  # force float32 output
feat_model = tf.keras.Model(inputs=base_model.input, outputs=gap)
feat_model.trainable = False


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [17]:
# ---------------- FEATURE CACHING ----------------
def extract_and_cache_features(generator, model, split_name, out_dir, workers=4, use_mp=True):
    fx = os.path.join(out_dir, f"{split_name}_features.npy")
    fy = os.path.join(out_dir, f"{split_name}_labels.npy")
    if os.path.exists(fx) and os.path.exists(fy):
        feats = np.load(fx); labels = np.load(fy)
        print(f"⚡ Loaded cached {split_name} features:", feats.shape)
        return feats.astype('float32'), labels
    n = generator.samples
    steps = int(np.ceil(n / generator.batch_size))
    feats = model.predict(generator, steps=steps, verbose=1) # Removed workers and use_multiprocessing
    labels = generator.classes
    feats = feats.astype('float32')  # scikit-learn friendly
    np.save(fx, feats); np.save(fy, labels)
    print(f"💾 Saved {split_name} features to cache:", fx)
    return feats, labels

print("🔎 Extracting (or loading) features with caching ...")
X_train, y_train = extract_and_cache_features(train_gen, feat_model, "train", OUT_DIR, PRED_WORKERS, PRED_USE_MP)
X_val,   y_val   = extract_and_cache_features(val_gen,   feat_model, "val",   OUT_DIR, PRED_WORKERS, PRED_USE_MP)
X_test,  y_test  = extract_and_cache_features(test_gen,  feat_model, "test",  OUT_DIR, PRED_WORKERS, PRED_USE_MP)

🔎 Extracting (or loading) features with caching ...


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


438/438 ━━━━━━━━━━━━━━━━━━━━ 71s 100ms/step
💾 Saved train features to cache: /content/drive/MyDrive/Colab Notebooks/RF_Finetune_Optimized_20250926_033531/train_features.npy
135/135 ━━━━━━━━━━━━━━━━━━━━ 29s 216ms/step
💾 Saved val features to cache: /content/drive/MyDrive/Colab Notebooks/RF_Finetune_Optimized_20250926_033531/val_features.npy
113/113 ━━━━━━━━━━━━━━━━━━━━ 30s 267ms/step
💾 Saved test features to cache: /content/drive/MyDrive/Colab Notebooks/RF_Finetune_Optimized_20250926_033531/test_features.npy


In [20]:
# ---------------- SINGLE RF SWEEP (for curve & baseline) ----------------
oob_scores, val_scores, tried_n = [], [], []
best_clf_rf, best_val_rf, best_n = None, -np.inf, None
no_improve = 0

for n_estimators in N_GRID:
    clf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=MAX_DEPTH,
        min_samples_split=MIN_SAMPLES_SPLIT,
        min_samples_leaf=MIN_SAMPLES_LEAF,
        max_features=MAX_FEATURES,
        class_weight="balanced",
        oob_score=True, bootstrap=True,
        random_state=SEED, n_jobs=-1
    )
    clf.fit(X_train, y_train)
    oob = clf.oob_score_
    y_val_pred = clf.predict(X_val)
    val_acc = accuracy_score(y_val, y_val_pred)

    oob_scores.append(oob); val_scores.append(val_acc); tried_n.append(n_estimators)
    print(f"Trees={n_estimators} | OOB={oob:.4f} | ValAcc={val_acc:.4f}")

    if val_acc > best_val_rf + 1e-4:
        best_clf_rf, best_val_rf, best_n = clf, val_acc, n_estimators
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f"⏹️ Early stop: no val improvement for {PATIENCE} steps.")
            break

print(f"✅ Best RF (sweep): n_estimators={best_n}, OOB={best_clf_rf.oob_score_:.4f}, ValAcc={best_val_rf:.4f}")


Trees=100 | OOB=0.4326 | ValAcc=0.4261
Trees=200 | OOB=0.4485 | ValAcc=0.4415
Trees=300 | OOB=0.4555 | ValAcc=0.4457
Trees=400 | OOB=0.4575 | ValAcc=0.4479
Trees=500 | OOB=0.4591 | ValAcc=0.4510
Trees=600 | OOB=0.4624 | ValAcc=0.4526
✅ Best RF (sweep): n_estimators=600, OOB=0.4624, ValAcc=0.4526


In [21]:
# Plot OOB & Val vs Trees
plt.figure(figsize=(8,6))
plt.plot(tried_n, oob_scores, marker='o', label='OOB Score')
plt.plot(tried_n, val_scores, marker='s', label='Validation Accuracy')
plt.title('OOB & Validation Accuracy vs n_estimators')
plt.xlabel('n_estimators'); plt.ylabel('Score'); plt.legend(); plt.grid(True)
plt.savefig(os.path.join(OUT_DIR, "oob_val_curve.png"), dpi=150, bbox_inches='tight'); plt.close()


In [22]:
# ---------------- MULTI-MODEL COMPARISON ----------------
# Try: tuned RF (depth=30), deep RF (None), LogReg+PCA95, XGBoost (if installed)
def eval_on_split(name, clf, Xtr, ytr, Xva, yva):
    clf.fit(Xtr, ytr)
    yva_pred = clf.predict(Xva)
    val_acc = accuracy_score(yva, yva_pred)
    return {"name": name, "model": clf, "val_acc": val_acc}

candidates = []


In [23]:
# RF — tuned (as requested)
rf_tuned = RandomForestClassifier(
    n_estimators=max(300, best_n or 300),  # ensure enough trees
    max_depth=MAX_DEPTH,
    min_samples_split=MIN_SAMPLES_SPLIT,
    min_samples_leaf=MIN_SAMPLES_LEAF,
    max_features=MAX_FEATURES,
    class_weight='balanced',
    oob_score=True, bootstrap=True,
    random_state=SEED, n_jobs=-1
)
candidates.append(eval_on_split("RF_tuned_d30_log2", rf_tuned, X_train, y_train, X_val, y_val))


In [24]:
# RF — deep/none (reduce underfit)
rf_deep = RandomForestClassifier(
    n_estimators=max(500, best_n or 500),
    max_depth=None,
    min_samples_split=4,
    min_samples_leaf=2,
    max_features='sqrt',
    class_weight='balanced',
    oob_score=True, bootstrap=True,
    random_state=SEED, n_jobs=-1
)
candidates.append(eval_on_split("RF_deep_none_sqrt", rf_deep, X_train, y_train, X_val, y_val))


In [25]:
# Logistic Regression — scaled + PCA 95%
from sklearn.linear_model import LogisticRegression
logreg_pipe = Pipeline([
    ("scaler", StandardScaler(with_mean=True, with_std=True)),
    ("pca", PCA(n_components=0.95, svd_solver="full")),
    ("clf", LogisticRegression(max_iter=2000, multi_class="multinomial",
                               solver="lbfgs", class_weight="balanced", n_jobs=-1))
])
candidates.append(eval_on_split("LogReg_PCA95", logreg_pipe, X_train, y_train, X_val, y_val))


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


In [26]:
# XGBoost (optional)
try:
    import xgboost as xgb
    xgb_clf = xgb.XGBClassifier(
        n_estimators=500, max_depth=6, learning_rate=0.05,
        subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
        objective="multi:softprob", num_class=num_classes,
        tree_method="hist", n_jobs=-1, random_state=SEED
    )
    candidates.append(eval_on_split("XGB_hist", xgb_clf, X_train, y_train, X_val, y_val))
    HAS_XGB = True
except Exception as e:
    HAS_XGB = False
    print("⚠️ xgboost not found; skipping XGB. (Optional) Install in Colab: !pip install xgboost")


In [27]:
# Pick best by validation accuracy
candidates = sorted(candidates, key=lambda d: d["val_acc"], reverse=True)
best = candidates[0]
best_name = best["name"]; best_clf = best["model"]; best_val = best["val_acc"]
print("🏆 Best by Val:", best_name, "| ValAcc=", round(best_val, 4))


🏆 Best by Val: XGB_hist | ValAcc= 0.4995


In [28]:
# ---------------- TEST EVAL + SAVE ARTIFACTS ----------------
y_test_pred = best_clf.predict(X_test)
test_acc = accuracy_score(y_test, y_test_pred)
print("🎯 TestAcc:", round(test_acc, 4))


🎯 TestAcc: 0.502


In [29]:
# Reports
report = classification_report(y_test, y_test_pred, target_names=classes_sorted, output_dict=True)
report_path_json = os.path.join(OUT_DIR, f"{best_name}_classification_report.json")
with open(report_path_json, "w") as f:
    json.dump(report, f, indent=2)


In [30]:
# Confusion matrix
cm = confusion_matrix(y_test, y_test_pred, labels=list(range(num_classes)))
np.save(os.path.join(OUT_DIR, f"{best_name}_confusion_matrix.npy"), cm)

plt.figure(figsize=(8,6))
plt.imshow(cm, interpolation='nearest')
plt.title(f'Confusion Matrix — {best_name}')
plt.colorbar()
tick = np.arange(num_classes)
plt.xticks(tick, classes_sorted, rotation=45, ha='right')
plt.yticks(tick, classes_sorted)
plt.xlabel('Predicted'); plt.ylabel('True'); plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, f"{best_name}_confusion_matrix.png"), dpi=150, bbox_inches='tight'); plt.close()


In [31]:
# Accuracy bars (Train/Val/Test)
train_acc = accuracy_score(y_train, best_clf.predict(X_train))
plt.figure(figsize=(6,5))
plt.bar(['Train', 'Val', 'Test'], [train_acc, best_val, test_acc])
plt.title(f'Accuracy (Train vs Val vs Test) — {best_name}')
plt.ylabel('Accuracy')
plt.savefig(os.path.join(OUT_DIR, f"{best_name}_accuracy_bar.png"), dpi=150, bbox_inches='tight'); plt.close()


In [32]:
# ROC for probabilistic models
def try_get_proba(clf, X):
    return clf.predict_proba(X) if hasattr(clf, "predict_proba") else None

y_proba = try_get_proba(best_clf, X_test)
if y_proba is not None:
    y_bin = label_binarize(y_test, classes=list(range(num_classes)))
    fpr, tpr, roc_auc = {}, {}, {}
    for i in range(num_classes):
        fpr[i], tpr[i], _ = roc_curve(y_bin[:, i], y_proba[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
    # micro/macro
    fpr["micro"], tpr["micro"], _ = roc_curve(y_bin.ravel(), y_proba.ravel())
    roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])
    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(num_classes)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(num_classes):
        mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
    mean_tpr /= num_classes
    fpr["macro"] = all_fpr; tpr["macro"] = mean_tpr
    roc_auc["macro"] = auc(fpr["macro"], tpr["macro"])

    plt.figure(figsize=(8,6))
    plt.plot(fpr["micro"], tpr["micro"], linestyle='--', label=f'micro-average ROC (AUC={roc_auc["micro"]:.3f})')
    plt.plot(fpr["macro"], tpr["macro"], linestyle='--', label=f'macro-average ROC (AUC={roc_auc["macro"]:.3f})')
    for i in range(num_classes):
        plt.plot(fpr[i], tpr[i], label=f'ROC {classes_sorted[i]} (AUC={roc_auc[i]:.3f})')
    plt.plot([0,1],[0,1],'k--'); plt.xlim([0,1]); plt.ylim([0,1.05])
    plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
    plt.title(f'One-vs-Rest ROC — {best_name}')
    plt.legend(loc='lower right', fontsize=8)
    plt.savefig(os.path.join(OUT_DIR, f"{best_name}_roc.png"), dpi=150, bbox_inches='tight'); plt.close()

# Save best model + meta
joblib.dump(best_clf, os.path.join(OUT_DIR, f"{best_name}.joblib"))

# Sweep summary (single RF sweep + multi-model)
import pandas as pd
pd.DataFrame([{"model":d["name"], "val_acc":d["val_acc"]} for d in candidates]) \
  .to_csv(os.path.join(OUT_DIR, "model_val_summary.csv"), index=False)

meta = {
    "backbone": BACKBONE,
    "img_size": IMG_SIZE,
    "n_classes": num_classes,
    "classes": classes_sorted,
    "train_samples": int(train_gen.samples),
    "val_samples": int(val_gen.samples),
    "test_samples": int(test_gen.samples),
    "rf_sweep": {
        "tried_n": tried_n,
        "oob_scores": [float(x) for x in oob_scores],
        "val_scores": [float(x) for x in val_scores],
        "best_n": int(best_n),
        "best_oob": float(best_clf_rf.oob_score_),
        "best_val_acc": float(best_val_rf),
    },
    "selected_model": {
        "name": best_name,
        "val_accuracy": float(best_val),
        "test_accuracy": float(test_acc),
    }
}
with open(os.path.join(OUT_DIR, "metadata.json"), "w") as f:
    json.dump(meta, f, indent=2)

print("\n📦 Saved files in:", OUT_DIR)
print(" - classification_report:", report_path_json)
print(" - confusion_matrix (npy/png), accuracy_bar, roc (if proba), oob_val_curve, model .joblib, metadata.json, model_val_summary.csv")
print("✅ Done.")


📦 Saved files in: /content/drive/MyDrive/Colab Notebooks/RF_Finetune_Optimized_20250926_033531
 - classification_report: /content/drive/MyDrive/Colab Notebooks/RF_Finetune_Optimized_20250926_033531/XGB_hist_classification_report.json
 - confusion_matrix (npy/png), accuracy_bar, roc (if proba), oob_val_curve, model .joblib, metadata.json, model_val_summary.csv
✅ Done.
